In [1]:
import pandas as pd
import os
import sys
import json
from pathlib import Path

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
SRC_PATH = REPO_ROOT / "src"

# Insert src at the front of sys.path so imports work
sys.path.insert(0, str(SRC_PATH))

In [2]:
import importlib
import preprocessing.icd_entity_extraction as iee

importlib.reload(iee)

<module 'preprocessing.icd_entity_extraction' from '/Users/brandonng/Documents/GitHub/ClinicalDigitalTwin/src/preprocessing/icd_entity_extraction.py'>

In [3]:
# Pass in_dir as the first argument
df = pd.read_csv('../data/processed/clinical_master.csv', dtype=str, low_memory=False)



In [4]:
from preprocessing.icd_entity_extraction import parse_icd_list

df['hosp_icd_codes_diagnosis_parsed'] = (
    df['hosp_icd_codes_diagnosis']
    .apply(parse_icd_list)
)

df['ed_icd_codes_diagnosis_parsed'] = (
    df['ed_icd_codes_diagnosis']
    .apply(parse_icd_list)
)

In [5]:
def normalize_icd(code):
    if code is None:
        return None
    if len(code) > 3 and "." not in code:
        return code[:3] + "." + code[3:]
    return code

normalize_icd('M79672')

'M79.672'

In [6]:
from preprocessing.icd_entity_extraction import icd_codes_to_descriptions
import icd10
# Getting descriptions of ICD codes for hosp and ed diagnoses

print(icd10.find('M79.672').description)
# 'Left ventricular failure'

df["hosp_icd_descriptions"] = df["hosp_icd_codes_diagnosis_parsed"].apply(
    icd_codes_to_descriptions
)

df.head()

Pain in left foot


,subject_id,hadm_id,ed_stay_id,ed_intime,ed_outtime,ed_icd_codes_diagnosis,ed_diagnosis,hosp_admittime,hosp_dischtime,race,...,death_time,hosp_icd_codes_diagnosis,hosp_diagnosis,icu_stay_id,icu_intime,icu_outtime,icu_count,hosp_icd_codes_diagnosis_parsed,ed_icd_codes_diagnosis_parsed,hosp_icd_descriptions
0,10049341,20677333.0,34255415.0,2171-04-07 17:48:00,2171-04-08 09:31:00,['78650'],['CHEST PAIN NOS'],2171-04-08 00:26:00,2171-04-08 09:31:00,ASIAN,...,NaN,"['78650', 'V103']","['Chest pain, unspecified', 'Personal history ...",NaN,NaN,NaN,NaN,"[78650, V103]",[78650],[person boarding or alighting a pedal cycle in...
1,10049341,NaN,35767475.0,2170-08-29 18:20:00,2170-08-29 22:46:00,"['78652', '7295', 'V103']","['PAINFUL RESPIRATION', 'PAIN IN LIMB', 'HX OF...",NaN,NaN,ASIAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[78652, 7295, V103]",[]
2,10049341,NaN,36382949.0,2171-11-19 20:09:00,2171-11-20 00:03:00,['7840'],['HEADACHE'],NaN,NaN,ASIAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[7840],[]
3,10049341,NaN,36490047.0,2174-11-29 19:39:00,2174-11-30 00:49:00,['M79672'],['Pain in left foot'],NaN,NaN,ASIAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[M79672],[]
4,10049341,NaN,37283116.0,2174-01-26 20:10:00,2174-01-27 00:34:00,"['M79671', 'W228XXA']","['Pain in right foot', 'Striking against or st...",NaN,NaN,ASIAN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[M79671, W228XXA]",[]


In [7]:
print(df.iloc[0]['hosp_diagnosis'])
print(df.iloc[0]['hosp_icd_descriptions'])

['Chest pain, unspecified', 'Personal history of malignant neoplasm of breast']
['person boarding or alighting a pedal cycle injured in collision with pedestrian or animal']


In [8]:
df["ed_icd_descriptions"] = df["ed_icd_codes_diagnosis_parsed"].apply(
    icd_codes_to_descriptions
)

In [9]:
df["ed_icd_descriptions"].iloc[1]

['person boarding or alighting a pedal cycle injured in collision with pedestrian or animal']

In [11]:
from preprocessing.icd_entity_extraction import filter_heart_codes
from preprocessing.icd_entity_extraction import icd_codes_to_descriptions


df['hosp_heart_icd_codes'] = df['hosp_icd_codes_diagnosis_parsed'].apply(filter_heart_codes)
df['ed_heart_icd_codes'] = df['ed_icd_codes_diagnosis_parsed'].apply(filter_heart_codes)

df['hosp_heart_descriptions'] = df['hosp_heart_icd_codes'].apply(icd_codes_to_descriptions)
df['ed_heart_descriptions'] = df['ed_heart_icd_codes'].apply(icd_codes_to_descriptions)

In [12]:
from preprocessing.icd_entity_extraction import map_row_to_labels


df['canonical_labels'] = (
    df['hosp_heart_descriptions'].apply(map_row_to_labels) +
    df['ed_heart_descriptions'].apply(map_row_to_labels)
)


In [13]:
from preprocessing.icd_entity_extraction import final_label_bucket


df['canonical_labels'] = df['canonical_labels'].apply(final_label_bucket)
df['canonical_labels'] = df['canonical_labels'].apply(lambda x: list(set(x)))

In [14]:
from preprocessing.icd_entity_extraction import canonical_map

for label in canonical_map.keys():
    df[label] = df['canonical_labels'].apply(lambda x: int(label in x))
df = df.drop(columns = ['canonical_labels'])

In [15]:
df["cardiovascular"] = df.apply(
    lambda row: "cardiovascular" if row["hosp_heart_icd_codes"] or row["ed_heart_icd_codes"] else "not cardiovascular",
    axis=1
)

In [ ]:
subset = df[df['ed_heart_icd_codes'].str.len() > 0]
subset[['hosp_diagnosis', 'ed_diagnosis']]

In [16]:
# Flatten heart descriptions for hospital + ED
all_heart_descs = pd.concat([
    df['hosp_heart_descriptions'].explode(),
    df['ed_heart_descriptions'].explode()
]).dropna().astype(str).str.lower()

heart_phrase_counts = all_heart_descs.value_counts()
MIN_COUNT = 50  # keep frequent enough phrases
heart_phrases_df = heart_phrase_counts[heart_phrase_counts >= MIN_COUNT].reset_index()
heart_phrases_df.columns = ['phrase', 'count']
heart_phrases_df.head(20)
#Used heart phrases to figure out map.

,phrase,count
0,atherosclerotic heart disease of native corona...,32895
1,unspecified atrial fibrillation,18588
2,old myocardial infarction,11916
3,paroxysmal atrial fibrillation,10057
4,chronic diastolic (congestive) heart failure,8492
5,acute on chronic diastolic (congestive) heart ...,6406
6,"heart failure, unspecified",5715
7,chronic systolic (congestive) heart failure,5679
8,non-st elevation (nstemi) myocardial infarction,4589
9,acute on chronic systolic (congestive) heart f...,4308


In [17]:
df.head()

,subject_id,hadm_id,ed_stay_id,ed_intime,ed_outtime,ed_icd_codes_diagnosis,ed_diagnosis,hosp_admittime,hosp_dischtime,race,...,bundle branch block,atrioventricular block,arrhythmias,pericardial disease,coronary artery disease,valvular disease,conduction disorder,cardiomegaly,other,cardiovascular
0,10049341,20677333.0,34255415.0,2171-04-07 17:48:00,2171-04-08 09:31:00,['78650'],['CHEST PAIN NOS'],2171-04-08 00:26:00,2171-04-08 09:31:00,ASIAN,...,0,0,0,0,0,0,0,0,0,not cardiovascular
1,10049341,NaN,35767475.0,2170-08-29 18:20:00,2170-08-29 22:46:00,"['78652', '7295', 'V103']","['PAINFUL RESPIRATION', 'PAIN IN LIMB', 'HX OF...",NaN,NaN,ASIAN,...,0,0,0,0,0,0,0,0,0,not cardiovascular
2,10049341,NaN,36382949.0,2171-11-19 20:09:00,2171-11-20 00:03:00,['7840'],['HEADACHE'],NaN,NaN,ASIAN,...,0,0,0,0,0,0,0,0,0,not cardiovascular
3,10049341,NaN,36490047.0,2174-11-29 19:39:00,2174-11-30 00:49:00,['M79672'],['Pain in left foot'],NaN,NaN,ASIAN,...,0,0,0,0,0,0,0,0,0,not cardiovascular
4,10049341,NaN,37283116.0,2174-01-26 20:10:00,2174-01-27 00:34:00,"['M79671', 'W228XXA']","['Pain in right foot', 'Striking against or st...",NaN,NaN,ASIAN,...,0,0,0,0,0,0,0,0,0,not cardiovascular


In [18]:
df.loc[:, ['subject_id', 'hadm_id', 'ed_stay_id'] + df.columns[df.columns.get_loc('heart failure'):].tolist()]

,subject_id,hadm_id,ed_stay_id,heart failure,myocardial infarction,atrial fibrillation,atrial flutter,cardiomyopathy,pulmonary hypertension,angina,bundle branch block,atrioventricular block,arrhythmias,pericardial disease,coronary artery disease,valvular disease,conduction disorder,cardiomegaly,other,cardiovascular
0,10049341,20677333.0,34255415.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
1,10049341,NaN,35767475.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
2,10049341,NaN,36382949.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
3,10049341,NaN,36490047.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
4,10049341,NaN,37283116.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530382,19912537,28640361.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
530383,13141797,22643367.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
530384,11286186,23566382.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular
530385,16578860,26155863.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,not cardiovascular


In [19]:
df['cardiovascular'].value_counts()


cardiovascular
not cardiovascular    452943
cardiovascular         77444
Name: count, dtype: int64